In [8]:
import sys
import os
import datasets
from datasets import Dataset, concatenate_datasets
from typing import List, Dict, Any
import random
import json

from transformers import AutoTokenizer

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [9]:
math_qa_questions = [
    "The school is planning a field trip. The school has 21 classrooms. There are 98 students in the school and more than 7 school buses. If some buses are full.How many seats are in each bus?",
    "Clayton plays basketball on a team. In the first game, he scored 10 points. In the second game, he scored 14 points. In the third game, he scored some points. In the fourth game, he scored the average of his points from the first four games.  How many total points did Clayton score during the first four games?",
    "Martha gets prize points every time she shops at her local grocery store. She gets 50 points per $10 spent, plus a 250 point bonus if she spends more than $100. Martha buys several pounds of beef for $11 each, less than 8 pounds of fruits and vegetables for $4/pound, 3 jars of spices for $6 each, and other groceries totaling $37. How many points does Martha get?",
    "When Michelle makes fresh pasta, she first makes the dough, then she rolls it out and cuts it, and then she hangs it on racks to dry for cooking later. She needs a drying rack for each three pounds of pasta she makes, and it takes two cups of flour to make each pound of pasta dough. How many more drying racks will Michelle need if she makes pasta using three 8-cup bags of flour?",
    "If John travels 0 miles on a bike ride, and Jill travels 5 miles less, how many miles does Jim travel if he travels only 20% as far as Jill?",
    "Every day Ryan spends 7 hours on learning english, 2 hours on learning Chinese and 4 hours on learning Spanish. How many more hours does he spend on learning english than he does on learning Japanese?",
    "Paul had at least 1000 books. After selling some in a garage sale .How many books did he sell?",
    "A magician was selling magic card decks for 5 dollars each. If he started with 14 decks and by the end of the day he had less than 5 left, how much money did he earn? ",
    "Mason is a caterer packing up silverware and plates for a big corporate event. Each piece of silverware weighs less than 4 ounces, and there are three pieces of silverware per setting. Each plate weighs 12 to 20 ounces each differently, and there are two plates per setting. If Mason needs enough settings for 15 tables with 8 settings each, plus 20 backup settings in case of breakage, how many ounces will all the settings weigh?",
    "Cassie is an athletic person and tries to drink at least 12 cups of water a day to stay hydrated while being active. Her water bottle holds 16 ounces. How many times does Cassie have to refill her water bottle a day to make sure she drinks 12 cups?",
    "Edward spent $ 16 to buy books and $ 8 to buy pens. Now he has $ 19..How much more did Edward spend on books than pens?",
    "Rachel had to complete 8 pages of math homework. If she had to complete 3 more pages of math homework than reading homework.How many pages did she have to complete in all?",
    "There were 30 women and 20 men who attended the party. After few hours, 2/5 of the total number of people left. If 9 men left the party, how many more women stayed at the party than men?",
    "There are 12 bananas and 4 apples in the blue basket. The red basket holds half as many fruits as the blue basket. How many fruits are in the red basket?",
    "Carol was sending out birthday invitations to her friends. If each package of invitations she bought had 9 invitations in it and she bought 5 packs,.how many friends can she invite?",
    "Lizzie has half as many crayons as Bobbie. Bobbie has three times as many crayons as Billie. If Billie has 18 crayons, how many crayons does Lizzie have?",
    "Lucca has 100 balls and 10 percent of his balls are basketballs. Lucien has 200 balls and 20 percent of them are basketballs. In total how many basketballs do Lucca and Lucien have?",
    "Aria has twice as many high school credits as Emily, who has twice the number of high school credits as Spencer. If Emily has 20 credits, what's twice the total number of high school credits the three have?",
    "A bag of dozen apples costs $14 and Brian has already spent $10 on kiwis and half that much on bananas. What's the maximum number of apples Brian can buy if he left his house with only $50 and needs to pay the $3.50 subway fare each way?",
    "The electricity price in Coco's town is $0.10 per kW. Coco's new oven has a consumption rate of 2.4 kWh (kilowatt-hours). How much will Coco pay for using his oven only if he used it for a total of 25 hours last month?",
    "Ember is half as old as Nate who is 14.  When she is 14 herself, how old will Nate be?",
]
print(len(math_qa_questions))

21


In [10]:
context_qa_questions = [
    "Who has the Islamic Liberation Party attempted to assassinate? ",
    "What was the team the Carolina Panthers played immediately prior to the NFC Championship game? ",
    "What was the score of the last game the Carolina Panthers played prior to the NFC Championship?",
    "Who did the Broncos beat to win their division in 2015?",
    "How many first downs did the Panthers have in Super Bowl 50?",
    "Who was the Panthers head coach for the 2015 season?",
    "What year was Casimir Pulaski born in Warsaw?",
    "What garden was formally only for royalty?",
    "How many natural reserves are in Warsaw?",
    "What was there a significant minority of in Warsaw?",
    "What is the name of the European Union agency for external border security?",
    "When was St. John's Cathedral constructed?",
    "When was a zoological garden established in the Praga Park?",
    "What part of France were the Normans located?",
    "What was the name of the leader ennobled by Henry III",
    "What disease did Tesla catch?",
    "Within what variable is L constrained according to the space hierarchy theorem?",
    "Who purhcased the remaining 4 pacakages available to broadcasters?",
    "How many plant species are estimated to be in the Amazon region?",
]
print(len(context_qa_questions))

19


In [4]:
math_data_path = "../../dataset/processed_data/test/base_UMWP_processed.json"
math_dataset = datasets.load_dataset("json", data_files=math_data_path)

math_data_path = "../../dataset/raw_model_responses/test/test_UMWP_trimmed.json"
base_model_math_dataset = datasets.load_dataset("json", data_files=math_data_path)

path = f"../../../dataset/raw_data/UMWP/test.json"
math_questions = datasets.load_dataset("json", data_files=path)

Generating train split: 0 examples [00:00, ? examples/s]

In [9]:
results = []
for sample in math_dataset["train"]:
    if sample["question"] in math_qa_questions:
        results.append(sample)

random.shuffle(results)
print(len(results))
print(results)

21
[{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a meticulous AI mathematician. Your task is to solve the following math problem.\n\nFollow these steps carefully:\n1. **Analyze the problem:** First, understand the given information and what is being asked.\n2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.\n3. **Solve or Explain:**\n   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.\n   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.\n\nYour entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nEdward spent $ 16 to bu

In [10]:
output_path = os.path.join(project_root, "base_model_math_test_results.json")
with open(output_path, "w") as f:
    json.dump(results, f, indent=4)

In [11]:
math_data_samples = []
for sample in math_questions["train"]:
    if sample["question"] in math_qa_questions:
        math_data_samples.append(sample)

random.shuffle(math_data_samples)
print(len(math_data_samples))
print(math_data_samples)

21
[{'question': 'When Michelle makes fresh pasta, she first makes the dough, then she rolls it out and cuts it, and then she hangs it on racks to dry for cooking later. She needs a drying rack for each three pounds of pasta she makes, and it takes two cups of flour to make each pound of pasta dough. How many more drying racks will Michelle need if she makes pasta using three 8-cup bags of flour?', 'answer': [0.0], 'answerable': False, 'source': 'GSM8K'}, {'question': 'Ember is half as old as Nate who is 14.  When she is 14 herself, how old will Nate be?', 'answer': [21.0], 'answerable': True, 'source': 'GSM8K'}, {'question': 'Paul had at least 1000 books. After selling some in a garage sale .How many books did he sell?', 'answer': [0.0], 'answerable': False, 'source': 'SVAMP'}, {'question': 'Edward spent $ 16 to buy books and $ 8 to buy pens. Now he has $ 19..How much more did Edward spend on books than pens?', 'answer': [8.0], 'answerable': True, 'source': 'SVAMP'}, {'question': 'The

In [12]:
output_path = os.path.join(project_root, "math_test_dataset.json")
with open(output_path, "w") as f:
    json.dump(math_data_samples, f, indent=4)

In [6]:
results = []
for sample in base_model_math_dataset["train"]:
    if sample["additional_info"]["question"] in math_qa_questions:
        results.append(sample)

random.shuffle(results)
print(len(results))
print(results)

21
[{'task_info': {'dataset': 'UMWP', 'type': 'QA'}, 'additional_info': {'answer': [45.0], 'answerable': True, 'domain': 'Math', 'model': 'Meta-Llama-3.1-8B-Instruct', 'question': 'Carol was sending out birthday invitations to her friends. If each package of invitations she bought had 9 invitations in it and she bought 5 packs,.how many friends can she invite?', 'source': 'ASDiv'}, 'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a meticulous AI mathematician. Your task is to solve the following math problem.\n\nFollow these steps carefully:\n1. **Analyze the problem:** First, understand the given information and what is being asked.\n2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.\n3. **Solve or Explain:**\n   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the 

In [7]:
output_path = os.path.join(project_root, "base_model_math_test_responses.json")
with open(output_path, "w") as f:
    json.dump(results, f, indent=4)

In [11]:
contex_data_path = "../../dataset/processed_data/test/base_rajpurkar_squad_processed.json"
context_dataset = datasets.load_dataset("json", data_files=contex_data_path)

math_data_path = "../../dataset/raw_model_responses/test/test_rajpurkar_squad_trimmed.json"
base_model_math_dataset = datasets.load_dataset("json", data_files=math_data_path)

path = f"../../../dataset/raw_data/rajpurkar_squad/test.parquet"
context_questions = datasets.load_dataset("parquet", data_files=path)

Generating train split: 0 examples [00:00, ? examples/s]

In [15]:
print(context_dataset)
print(context_questions)

DatasetDict({
    train: Dataset({
        features: ['input', 'question', 'context', 'answer', 'responses', 'errors', 'wrong_response_number'],
        num_rows: 10554
    })
})
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [16]:
results = []
for sample in context_dataset["train"]:
    if sample["question"] in context_qa_questions:
        results.append(sample)

random.shuffle(results)
print(len(results))
print(results)

19
[{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a specialized question-answering AI. Your task is to give a concise answer to the question using *only* the provided context. Make sure to always give an answer.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContext:\n'''\nOne of the most famous people born in Warsaw was Maria Skłodowska-Curie, who achieved international recognition for her research on radioactivity and was the first female recipient of the Nobel Prize. Famous musicians include Władysław Szpilman and Frédéric Chopin. Though Chopin was born in the village of Żelazowa Wola, about 60 km (37 mi) from Warsaw, he moved to the city with his family when he was seven months old. Casimir Pulaski, a Polish general and hero of the American Revolutionary War, was born here in 1745.\n'''\n\nQuestion: What year was Casimir Pulaski born in Warsaw?<|eot_id|><|start_header_id|>assistant<|end_header_id|>", 'question': 'What year was Casimir Pulas

In [17]:
output_path = os.path.join(project_root, "base_model_context_test_results.json")
with open(output_path, "w") as f:
    json.dump(results, f, indent=4)

In [18]:
context_data_samples = []
for sample in context_questions["train"]:
    if sample["question"] in context_qa_questions:
        context_data_samples.append(sample)

random.shuffle(context_data_samples)
print(len(context_data_samples))
print(context_data_samples)

19
[{'id': '56dde27d9a695914005b9652', 'title': 'Normans', 'context': 'The descendants of Rollo\'s Vikings and their Frankish wives would replace the Norse religion and Old Norse language with Catholicism (Christianity) and the Gallo-Romance language of the local people, blending their maternal Frankish heritage with Old Norse traditions and customs to synthesize a unique "Norman" culture in the north of France. The Norman language was forged by the adoption of the indigenous langue d\'oïl branch of Romance by a Norse-speaking ruling class, and it developed into the regional language that survives today.', 'question': 'What part of France were the Normans located?', 'answers': {'text': ['north', 'the north', 'north'], 'answer_start': [327, 323, 327]}}, {'id': '56d709ef0d65d21400198306', 'title': 'Super_Bowl_50', 'context': "With Rivera having been a linebacker with the Chicago Bears in Super Bowl XX, and Kubiak replacing Elway at the end of the Broncos' defeats in Super Bowls XXI and X

In [19]:
output_path = os.path.join(project_root, "context_test_dataset.json")
with open(output_path, "w") as f:
    json.dump(context_data_samples, f, indent=4)

In [12]:
results = []
for sample in base_model_math_dataset["train"]:
    if sample["additional_info"]["question"] in context_qa_questions:
        results.append(sample)

random.shuffle(results)
print(len(results))
print(results)

19
[{'task_info': {'dataset': 'rajpurkar_squad', 'type': 'Contextual QA'}, 'additional_info': {'answer': ['14th century', '14th century', '14th century'], 'context': 'Gothic architecture is represented in the majestic churches but also at the burgher houses and fortifications. The most significant buildings are St. John\'s Cathedral (14th century), the temple is a typical example of the so-called Masovian gothic style, St. Mary\'s Church (1411), a town house of Burbach family (14th century), Gunpowder Tower (after 1379) and the Royal Castle Curia Maior (1407–1410). The most notable examples of Renaissance architecture in the city are the house of Baryczko merchant family (1562), building called "The Negro" (early 17th century) and Salwator tenement (1632). The most interesting examples of mannerist architecture are the Royal Castle (1596–1619) and the Jesuit Church (1609–1626) at Old Town. Among the first structures of the early baroque the most important are St. Hyacinth\'s Church (16

In [13]:
output_path = os.path.join(project_root, "base_model_context_test_responses.json")
with open(output_path, "w") as f:
    json.dump(results, f, indent=4)